## Computational Verification I

This notebook provides computational verification of the formulas for the polynomial maps $\Theta$, $\Psi$, and $K$ given in Section~5. 
All computations are carried out over $\mathbb{Q}(i)$ using SageMath.

### 1. Quadratic line complexes (Proposition 5.11)

In [1]:
# Polynomial ring
K = QQ
S = PolynomialRing(K,names=('a1','a2','a3','a4','x1','x2','x3','x4','y1','y2','y3','y4'))
a1,a2,a3,a4,x1,x2,x3,x4,y1,y2,y3,y4 = S.gens()
# Plucker coordinates
p12 = x1*y2 - x2*y1
p13 = x1*y3 - x3*y1
p14 = x1*y4 - x4*y1
p23 = x2*y3 - x3*y2
p24 = x2*y4 - x4*y2
p34 = x3*y4 - x4*y3

# Plucker relation
assert (p12*p34 - p13*p24 + p14*p23) == 0

# Define gamma_a(z) 

def gamma(z1, z2, z3, z4):

    E1 = -((a1^2+a2^2)*z2*z4 + 2*a3*a4*z1*z3); F1 =  ((a1^2+a2^2)*z1*z3 + 2*a3*a4*z2*z4)
    E2 = -(a2*a3*z1^2 - a1*a4*z2^2- a1*a3*z3^2 + a2*a4*z4^2); F2 =  (a1*a4*z1^2 - a2*a3*z2^2- a2*a4*z3^2 + a1*a3*z4^2)
    E3 = -(a1*a3*z1^2 - a2*a4*z2^2+ a2*a3*z3^2 - a1*a4*z4^2); F3 = -(a2*a4*z1^2 - a1*a3*z2^2+ a1*a4*z3^2 - a2*a3*z4^2)
    E4 = -((a3^2+a4^2)*z1*z4 + 2*a1*a2*z2*z3); F4 = -((a3^2+a4^2)*z2*z3 + 2*a1*a2*z1*z4)
    E5 =  (a1^2-a2^2)*z1*z2 - (a3^2-a4^2)*z3*z4; F5 =  (a1^2-a2^2)*z3*z4 + (a3^2-a4^2)*z1*z2
    return (
        [E1,E2,E3,E4,E5],
        [F1,F2,F3,F4,F5]
    )



gamma_x = gamma(x1,x2,x3,x4)
gamma_y = gamma(y1,y2,y3,y4)

# define paring on V10

def pairing(u, v):
    Eu, Fu = u
    Ev, Fv = v

    return sum(
        Eu[j]*Fv[j] + Fu[j]*Ev[j]
        for j in range(5))

theta_from_gamma = S(pairing(gamma_x, gamma_y))

# Formula for Theta given in Prop 5.11
theta_formula = ((a1^2-a2^2)*(a3^2-a4^2)*(p12^2-p34^2)- 2*(a1^2+a2^2)*a3*a4*(p13^2+p24^2)+ (1/3)*(2*a1^4 + 2*a2^4 - a3^4 - a4^4+ 6*a3^2*a4^2)*(p12*p34-p14*p23)
+ 2*a1*a2*(a3^2+a4^2)*(p14^2+p23^2)+ (1/3)*(2*a3^4 + 2*a4^4- a1^4 - a2^4+ 6*a1^2*a2^2)*(p12*p34+p13*p24))

# Verify the formula for Theta
assert theta_from_gamma == -theta_formula;print("The formula for Theta is verified.")


The formula for Theta is verified.


### 2. Coble quadrics (Section 5.5)

We verify that, in the proof of Lemma 5.18, the following equality holds:
$$
AC-B^2 = x^2\theta^2 \cdot \Gamma(\Theta)
$$
modulo the ideal generated by $S+T+U$ and $x\xi + y\eta + z\zeta + t\theta$.

In [2]:
# Define the polynomial ring.

Ring = PolynomialRing(K,names=('P','Q','R','S','T','U','x','y','z','t','xi','eta','zeta','theta'))

P, Q, R, S, T, U, x, y, z, t, xi, eta, zeta, theta = Ring.gens()

# The incidence relation defining Fl(1,3;4), for ([x, y, z, t],[eta, zeta, theta]) on Fl(1,3;4):

h = x*xi + y*eta + z*zeta + t*theta

# Define the quotient ring by the two relations: S+T+U=0 on Gr(2,4) and h=0 on Fl(1,3;4).

relations = Ring.ideal([S+T+U, h])
QuotientRing = Ring.quotient(relations)

# Compute the Plücker coordinates of v1 wedge v2 in the order (12,13,14,23,24,34).

from itertools import combinations
pairs = list(combinations(range(4), 2))
def wedge_coords(v1, v2):
    return [
        v1[i]*v2[j] - v1[j]*v2[i]
        for i, j in pairs]

# local frame of U1 \wedge U3=<u,w> on Fl(1,3;4) when x*theta \not=0

v = [x, y, z, t]
a = [0, theta, 0, -eta]
b = [0, 0, theta, -zeta]
u = wedge_coords(v, a)
w = wedge_coords(v, b)
assert u == [x*theta, 0, -x*eta,-z*theta, -y*eta-t*theta, -z*eta]
assert w == [0, x*theta, -x*zeta,y*theta, -y*zeta, -z*zeta-t*theta]


# Define Theta and compute its restriction to U1 \wedge U3. We write Theta(s*u+r*w)=A*s^2+2*B*s*r+C*r^2.

def Theta(p):
    p12, p13, p14, p23, p24, p34 = p
    return (P*(p12**2-p34**2)+ Q*(p13**2+p24**2)+ R*(p14**2+p23**2)+ 2*S*p12*p34+ 2*T*p14*p23- 2*U*p13*p24)

# restricting Theta on the local frame and define A,B,C by the formula Theta(s*u+r*w)=A*s^2+2*B*s*r+C*r^2.

A = Theta(u); C = Theta(w); u_plus_w = [ui+wi for ui, wi in zip(u, w)]; B = (1/2)*(Theta(u_plus_w)-A-C)

# The local equation of the degeneracy locus, which is divided by (x*theta)^2 by definition.

disc_local = A*C-B**2

# define Gamma(Theta) by the nine quadratic coefficient functions and polynomials G_i provided in the Lemma.

b_coeffs = [P*Q, P*R, Q*R,P*(T-U),Q*(S-T),R*(S-U),Q**2+R**2-(T-U)**2,-P**2+R**2-(S-T)**2,-P**2+Q**2-(S-U)**2]

G = [x**2*theta**2 + y**2*zeta**2- z**2*eta**2 - t**2*xi**2, x**2*zeta**2 + y**2*theta**2- z**2*xi**2 - t**2*eta**2,
   x**2*eta**2 + y**2*xi**2+ z**2*theta**2 + t**2*zeta**2,-2*x*y*zeta*theta + 2*z*t*xi*eta,-2*x*z*eta*theta - 2*y*t*xi*zeta,
   2*x*t*eta*zeta + 2*y*z*xi*theta,x**2*xi**2 + x*z*xi*zeta+ x*t*xi*theta - z*t*zeta*theta,-x*z*xi*zeta + x*t*xi*theta
    + z*t*zeta*theta + t**2*theta**2,x*z*xi*zeta - x*t*xi*theta+ z**2*zeta**2 + z*t*zeta*theta]

Gamma_Theta = sum((bi*Gi for bi, Gi in zip(b_coeffs, G)),Ring.zero())

# Verify that A*C-B^2 = x^2*theta^2*Gamma_Theta modulo S+T+U=0 and h=0.

difference = disc_local - x**2*theta**2*Gamma_Theta

assert QuotientRing(difference) == 0

print("Verified: A*C-B^2 is congruent to " "x^2*theta^2*Gamma_Theta")

Verified: A*C-B^2 is congruent to x^2*theta^2*Gamma_Theta


### 3. Kummer quartic surfaces (Lemma 5.21 and Prop 5.22).

#### (3.1) Formula in Lemma 5.21

In [3]:
newRing=PolynomialRing(QQ,names=('P','Q','R','S','T','x','y','z','t','p12','p13','p23',a1,a2,a3,a4))
P, Q, R, S, T, x, y, z, t, p12, p13, p23, a1, a2, a3, a4 = newRing.gens()
p14=y*p12+z*p13; p24=-x*p12+z*p23; p34=-x*p13-y*p23
Theta=P*(p12**2-p34**2)+ Q*(p13**2+p24**2)+ R*(p14**2+p23**2)+ 2*S*p12*p34+ 2*T*p14*p23
variables=(p12,p13,p23)
M=(1/2)*matrix([[Theta.derivative(x).derivative(y) for y in variables] for x in variables])
K=det(M)(x=x/t,y=y/t,z=z/t,t=1)*t^4;
Keq=-P*Q*R*(x^4+y^4-z^4-t^4)-P*(Q^2+R^2-T^2)*(x^2*y^2-z^2*t^2)+Q*(R^2-P^2-(S-T)^2)*(x^2*z^2+y^2*t^2)+R*(Q^2-P^2-S^2)*(x^2*t^2+y^2*z^2)+2*(S*(R^2-Q^2)+T*(P^2+Q^2)+S*T*(S-T))*x*y*z*t
assert K == Keq; 
print("Verified: det(M) is equal to the formula in Lemma 5.20")

Verified: det(M) is equal to the formula in Lemma 5.20


#### (3.2). Formula for $A(a),B(a),C(a),D(a),E(a)$ in Prop 5.22

In [4]:
Pn=(a1^2-a2^2)*(a3^2-a4^2)
Qn=-2*(a1^2+a2^2)*a3*a4
Rn=2*a1*a2*(a3^2+a4^2)
Sn=(1/2)*((a3^2+a4^2)^2+4*a1^2*a2^2)
Tn=(1/2)*((a3^2-a4^2)^2-(a1^2-a2^2)^2)
A=-Pn*Qn*Rn
B=-Pn*(Qn^2+Rn^2-Tn^2)
C=Qn*(Rn^2-Pn^2-(Sn-Tn)^2)
D=Rn*(Qn^2-Pn^2-Sn^2)
E=Sn*(Rn^2-Qn^2)+Tn*(Pn^2+Qn^2)+Sn*Tn*(Sn-Tn)
print(A)
print(B)
print(C)
print(D)        
print(E)

4*a1^5*a2*a3^5*a4 - 4*a1*a2^5*a3^5*a4 - 4*a1^5*a2*a3*a4^5 + 4*a1*a2^5*a3*a4^5
1/4*a1^10*a3^2 - 5/4*a1^8*a2^2*a3^2 + 5/2*a1^6*a2^4*a3^2 - 5/2*a1^4*a2^6*a3^2 + 5/4*a1^2*a2^8*a3^2 - 1/4*a2^10*a3^2 - 1/2*a1^6*a3^6 - 5/2*a1^4*a2^2*a3^6 + 5/2*a1^2*a2^4*a3^6 + 1/2*a2^6*a3^6 + 1/4*a1^2*a3^10 - 1/4*a2^2*a3^10 - 1/4*a1^10*a4^2 + 5/4*a1^8*a2^2*a4^2 - 5/2*a1^6*a2^4*a4^2 + 5/2*a1^4*a2^6*a4^2 - 5/4*a1^2*a2^8*a4^2 + 1/4*a2^10*a4^2 - 5/2*a1^6*a3^4*a4^2 - 25/2*a1^4*a2^2*a3^4*a4^2 + 25/2*a1^2*a2^4*a3^4*a4^2 + 5/2*a2^6*a3^4*a4^2 - 5/4*a1^2*a3^8*a4^2 + 5/4*a2^2*a3^8*a4^2 + 5/2*a1^6*a3^2*a4^4 + 25/2*a1^4*a2^2*a3^2*a4^4 - 25/2*a1^2*a2^4*a3^2*a4^4 - 5/2*a2^6*a3^2*a4^4 + 5/2*a1^2*a3^6*a4^4 - 5/2*a2^2*a3^6*a4^4 + 1/2*a1^6*a4^6 + 5/2*a1^4*a2^2*a4^6 - 5/2*a1^2*a2^4*a4^6 - 1/2*a2^6*a4^6 - 5/2*a1^2*a3^4*a4^6 + 5/2*a2^2*a3^4*a4^6 + 5/4*a1^2*a3^2*a4^8 - 5/4*a2^2*a3^2*a4^8 - 1/4*a1^2*a4^10 + 1/4*a2^2*a4^10
1/2*a1^10*a3*a4 + 5/2*a1^8*a2^2*a3*a4 + 5*a1^6*a2^4*a3*a4 + 5*a1^4*a2^6*a3*a4 + 5/2*a1^2*a2^8*a3*a4 + 1/2*a2^10*